In [0]:
%run ./01-config

In [0]:
class SetupHelper():
    
    def __init__(self, env):
        conf = Config()
        self.landing_zone = conf.base_dir_data + "/raw"
        self.checkpoint_zone = conf.base_dir_checkpoint + "/checkpoints"
        self.catalog = env
        self.bronze_db = conf.bronze_db
        self.silver_db = conf.silver_db
        self.gold_db = conf.gold_db

        # historical data
        self.create_date_lookup(self.silver_db)

        # bronze layer
        self.create_db(self.bronze_db)
        self.create_bronze_tables(self.bronze_db)

        # silver layer
        self.create_db(self.silver_db)
        self.create_silver_tables(self.silver_db)

        # gold layer
        self.create_db(self.gold_db)
        self.create_gold_tables(self.gold_db, self.bronze_db, self.silver_db)

        self.use_db(self.bronze_db)

        self.initialized = False

    def create_db(self, db_name):
        spark.catalog.clearCache()

        # setup bronze database
        print(f"Creating database {self.catalog}.{db_name}...", end='')
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.catalog}.{db_name}")
    
        self.initialized = True
        print("Done")
    
    def use_db(self, db_name):
        spark.catalog.clearCache()
        spark.sql(f"USE {self.catalog}.{db_name}")

    def create_date_lookup(self, db_name):
        print(f"Creating table date_lookup in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.date_lookup (
                date date,
                week int,
                year int,
                month int,
                day_of_week int,
                day_of_month int,
                day_of_year int,
                week_part string)
            """)
        print("Done")
    
    def create_bronze_tables(self, db_name):
        print(f"Creating tables in {self.catalog}.{db_name}...", end='')
        if (self.initialized):
            print(f"Creating bronze tables in {self.catalog}.{db_name}...", end='')
            self.create_registered_users_bz(db_name)
            self.create_gym_logins_bz(db_name)
            self.create_kafka_multiplex_bz(db_name)
        else:
            raise ReferenceError(f"Database {self.catalog}.{db_name} not initialized. Cannot create bronze tables in database.")

    def create_registered_users_bz(self, db_name):
        print(f"Creating table registered_users_bz in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.registered_users_bz (
                user_id long,
                device_id long,
                mac_address string,
                registration_timestamp double,
                load_time timestamp,
                source_file string
            )
        """)
        print("Done")
    
    def create_gym_logins_bz(self, db_name):
        print(f"Creating table gym_logins_bz in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.gym_logins_bz (
                mac_address string,
                gym bigint,
                login double,
                logout double,
                load_time timestamp,
                source_file string
            )
        """)
        print("Done")
    
    def create_kafka_multiplex_bz(self, db_name):
        print(f"Creating table kafka_multiplex_bz in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.kafka_multiplex_bz (
                key string,
                value string,
                topic string,
                partition int,
                offset bigint,
                timestamp double,
                date date,
                week_part string,
                load_time timestamp,
                source_file string
            )
            PARTITIONED BY (topic, week_part)
        """)
        print("Done")

    def create_silver_tables(self, db_name):
        print(f"Creating tables in {self.catalog}.{db_name}...", end='')
        if (self.initialized):
            print(f"Creating silver tables in {self.catalog}.{db_name}...", end='')
            self.create_users(db_name)
            self.create_gym_logs(db_name)
            self.create_user_profile(db_name)
            self.create_heart_rate(db_name)
            self.create_user_bins(db_name)
            self.create_workouts(db_name)
            self.create_completed_workouts(db_name)
            self.create_workout_bpm(db_name)
        else:
            raise ReferenceError(f"Database {self.catalog}.{db_name} not initialized. Cannot create silver tables in database.")
    
    def create_users(self, db_name):
        print(f"Creating table users in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.users (
                user_id int,
                device_id int,
                mac_address string,
                timestamp timestamp,
            )
        """)
        print("Done")
    
    def create_gym_logs(self, db_name):
        print(f"Creating table gym_logs in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.gym_logs (
                mac_address string,
                gym bigint,
                login timestamp,
                logout timestamp
            )
        """)
        print("Done")
    
    def create_user_profile(self, db_name):
        print(f"Creating table user_profile in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.user_profile (
                user_id int,
                dob date,
                gender string,
                first_name string,
                last_name string,
                street_address string,
                city string,
                state string,
                zip int,
                updated timestamp)
            """)
        print("Done")
    
    def create_heart_rate(self, db_name):
        print(f"Creating table heart_rate in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.heart_rate (
                device_id int,
                time timestamp,
                heartrate double,
                valid boolean)
            """)
        print("Done")
    
    def create_user_bins(self, db_name):
        print(f"Creating table user_bins in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.user_bins (
                user_id int,
                age string,
                gender string,
                city string,
                state string)
            """)
        print("Done")
    
    def create_workouts(self, db_name):
        print(f"Creating table workouts in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.workouts (
                user_id int,
                workout_id int,
                session_id int,
                time timestamp,
                action string
            )
            """)
        print("Done")
    
    def create_completed_workouts(self, db_name):
        print(f"Creating table completed_workouts in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.completed_workouts (
                user_id int,
                workout_id int,
                session_id int,
                start_time timestamp,
                end_time timestamp
            )
        """)
        print("Done")
    
    def create_workout_bpm(self, db_name):
        print(f"Creating table workout_bpm in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.workout_bpm (
                user_id int,
                workout_id int,
                session_id int,
                start_time timestamp,
                end_time timestamp,
                heartrate double
            )
        """)
    
    def create_gold_tables(self, db_name, bronze_db_name, silver_db_name):
        print(f"Creating tables in {self.catalog}.{db_name}...", end='')
        if (self.initialized):
            self.create_workout_bpm_summary(db_name)
            self.create_gym_summary(db_name, bronze_db_name, silver_db_name)
        else:
            raise ReferenceError(f"Database {self.catalog}.{db_name} not initialized. Cannot create gold tables in database.")
    
    def create_workout_bpm_summary(self, db_name):
        print(f"Creating table workout_bpm_summary in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.catalog}.{db_name}.workout_bpm_summary (
                workout_id int,
                session_id int,
                user_id int,
                age string,
                gender string,
                city string,
                state string,
                min_bpm double,
                avg_bpm double,
                max_bpm double,
                num_recordings int
            )
        """)
        print("Done")
    
    def create_gym_summary(self, db_name, bronze_db_name, silver_db_name):
        print(f"Creating gold view gym_summary in {self.catalog}.{db_name}...", end='')
        spark.sql(f"""
            CREATE OR REPLACE VIEW {self.catalog}.{db_name}.gym_summary AS
            SELECT
                to_date(login::timestamp) date,
                gym,
                l.mac_address,
                workout_id,
                session_id,
                round((logout::long - login::long)/60,2) minutes_in_gym,
                round((end_time::long - start_time::long)/60,2) minutes_excercising
            FROM {self.catalog}.{silver_db_name}.gym_logs l
            JOIN (
                SELECT mac_address, workout_id, session_id, start_time, end_time
                FROM {self.catalog}.{silver_db_name}.completed_workouts w
                INNER JOIN {self.catalog}.{silver_db_name}.users u on w.user_id = u.user_id
            ) uw ON l.mac_address = uw.mac_address AND uw.start_time between l.login and l.logout
            ORDER BY date, gym, l.mac_address, session_id
        """)
        print("Done")

    def assert_table(self, db_name, table_name):
        assert spark.sql(f"SHOW TABLES IN {self.catalog}.{db_name}") \
                .filter(f"isTemporary == false and tableName == '{table_name}'") \
                .count() == 1, f"Table {table_name} not found in {self.catalog}.{db_name}"
        print("Table {table_name} found in {self.catalog}.{db_name}")
    
    def assert_view(self, db_name, view_name):
        assert spark.sql(f"SHOW VIEWS IN {self.catalog}.{db_name}") \
                .filter(f"isTemporary == false and tableName == '{view_name}'") \
                .count() == 1, f"View {view_name} not found in {self.catalog}.{db_name}"
        print("View {view_name} found in {self.catalog}.{db_name}")
    
    def validate(self):
        # validate bronze layer
        self.assert_table(self.bronze_db, "registered_users_bz")
        self.assert_table(self.bronze_db, "gym_logins_bz")
        self.assert_table(self.bronze_db, "kafka_multiplex_bz")
        
        # validate silver layer
        self.assert_table(self.silver_db, "users")
        self.assert_table(self.silver_db, "user_bins")
        self.assert_table(self.silver_db, "workouts")
        self.assert_table(self.silver_db, "completed_workouts")
        self.assert_table(self.silver_db, "workout_bpm")
        self.assert_table(self.silver_db, "date_lookup")
        self.assert_table(self.silver_db, "gym_logs")

        # validate gold layer
        self.assert_view(self.gold_db, "gym_summary")
        self.assert_table(self.gold_db, "workout_bpm_summary")
    
    def cleanup(self):
        # drop all three databases
        if spark.sql(f"SHOW DATABASES IN {self.catalog}").count() > 0:
            spark.sql(f"DROP DATABASE IF EXISTS {self.catalog}.{self.bronze_db} CASCADE")
            spark.sql(f"DROP DATABASE IF EXISTS {self.catalog}.{self.silver_db} CASCADE")
            spark.sql(f"DROP DATABASE IF EXISTS {self.catalog}.{self.gold_db} CASCADE")
        # clean out landing zone
        dbutils.fs.rm(self.landing_zone, True)
        # clean out checkpoint zone
        dbutils.fs.rm(self.checkpoint_zone, True)
        print("Cleanup complete")





    